# TTK4115 — Lab 4: Kalman filter

Post-processing of the recorded helicopter experiments (Part IV of the lab assignment).

The notebook follows **Task 1 (noise), Task 2 (implementation) and Task 3 (experimentation)**. Figures are exported to `../Plots/PNG` and `../Plots/PDF`, ready for inspection and inclusion in the lab report.

**Important:** The original MAT files are not modified. Encoder channels are reordered before comparison, and a *documented +0.60 rad correction* is applied to the logged elevation encoder signal (double offset in the supplied Simulink model). The correction does not remove the remaining IMU–encoder bias.

Missing recordings (ground noise, new-data switch, and covariance evolution) are not reconstructed.

## 0. Setup and recorded data

Only NumPy, SciPy and Matplotlib are required. Run cells from top to bottom. The notebook also works when launched from the project root instead of its own folder.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

# Use the notebook folder, whether Jupyter was launched here or from the ZIP root.
BASE = Path.cwd()
if not (BASE / "input").is_dir():
    BASE = BASE / "Notebook_og_input"
INPUT = BASE / "input"
PLOT_DIR = BASE.parent / "Plots"
(PLOT_DIR / "PNG").mkdir(parents=True, exist_ok=True)
(PLOT_DIR / "PDF").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180,
                     "axes.grid": True, "grid.alpha": 0.25})

# Q values are interpreted from filenames; the actual lab-workspace values
# were not archived and cannot be verified from the provided files.
CASES = [("p00001", 0.00001), ("p0001", 0.0001),
         ("p01", 0.1), ("1", 1.0), ("20", 20.0)]
STATE_NAMES = ["Pitch", "Pitch rate", "Elevation", "Elevation rate",
               "Travel", "Travel rate"]
STATE_LABELS = [r"$p$ [rad]", r"$\dot p$ [rad/s]",
                r"$e$ [rad]", r"$\dot e$ [rad/s]",
                r"$\lambda$ [rad]", r"$\dot\lambda$ [rad/s]"]


def save_figure(fig, name):
    fig.savefig(PLOT_DIR / "PNG" / (name + ".png"), bbox_inches="tight")
    fig.savefig(PLOT_DIR / "PDF" / (name + ".pdf"), bbox_inches="tight")
    plt.show()

### Signal mapping

Each MAT file contains `ans` with time in the **first row** and signals in the following rows. From the supplied Simulink connections:

| Recording | Signal order after time |
|---|---|
| `feedback_xhat` and `feedback_yvalues` | $[p,\dot p,e,\dot e,\lambda,\dot\lambda]$ |
| `feedback_encodervalues` (raw saved order) | $[\lambda,\dot\lambda,p,\dot p,e,\dot e]$ |
| `reference` | pitch reference $p_c$, elevation-rate reference $\dot e_c$ |

The travel-angle measurement in `y` is **directly supplied by the encoder** in the Simulink `y-transformer`; it is not an independent IMU measurement. This is why travel in `y` and the encoders coincides.

For the saved elevation encoder signal, a `0.60 rad` offset is subtracted **inside `Heli 3D`** and **again before logging**. The comparison below adds back the extra `0.60 rad`, but leaves the remaining sensor discrepancy visible.

In [ ]:
def read_mat(filename):
    data = loadmat(INPUT / filename)["ans"]
    return np.asarray(data, dtype=float)


def load_run(suffix):
    stem = "task44_"
    xhat = read_mat(f"{stem}feedback_xhatqis{suffix}.mat")
    y = read_mat(f"{stem}feedback_yvaluesqis{suffix}.mat")
    enc_raw = read_mat(f"{stem}feedback_encodervaluesqis{suffix}.mat")
    reference = read_mat(f"{stem}reference_qis{suffix}.mat")

    # Align encoder channels with [p, p_dot, e, e_dot, lambda, lambda_dot].
    encoder = enc_raw[[3, 4, 5, 6, 1, 2], :].copy()
    encoder[2] += 0.60  # Remove ONLY the additional logging offset.
    return {"t": xhat[0], "xhat": xhat[1:], "y": y[1:],
            "encoder": encoder, "encoder_raw": enc_raw[1:],
            "reference": reference[1:], "time_arrays": [xhat[0], y[0],
                                                 enc_raw[0], reference[0]]}

runs = {suffix: load_run(suffix) for suffix, _ in CASES}

### Basic checks

Check shape, missing values, sampling step and timestamp alignment. These checks are diagnostic: they do not alter the recordings.

In [ ]:
print(f"{'Case':<12} {'Samples':>9} {'Duration [s]':>14} {'dt [s]':>9}  Status")
for suffix, q in CASES:
    run = runs[suffix]
    t = run["t"]
    arrays = [run["xhat"], run["y"], run["encoder"], run["reference"]]
    time_ok = all(np.array_equal(t, ti) for ti in run["time_arrays"])
    finite = all(np.isfinite(a).all() for a in arrays)
    dt_ok = np.allclose(np.diff(t), 0.002, atol=1e-10)
    shapes_ok = (all(a.shape == (6, t.size) for a in arrays[:3])
                 and arrays[3].shape == (2, t.size))
    ok = time_ok and finite and dt_ok and shapes_ok
    print(f"{q:<12g} {t.size:9d} {t[-1]:14.3f} {np.median(np.diff(t)):9.3f}  "
          f"{'OK' if ok else 'CHECK DATA'}")
    assert ok, f"Invalid recording: {suffix}"

## 1. Task 1 — Measurement noise and $R_d$

The only supplied Task 1 recording is `task41_xhat_noie_fligh.mat` (19.84 s). The MATLAB initialization computes `Rd = cov(X_meas')` using this file, i.e. the sample covariance of **all six channels**. The name includes `xhat`, so the file alone does not prove that all channels are unfiltered measurements. The original lab workspace/scope routing for this recording is not included.

There is **no stationary-on-ground recording**, so the two requested experimental noise covariances cannot be compared. Drift or movement in this flying recording can also inflate the covariance; treat the matrix as a *recording covariance*, not necessarily a pure sensor-noise covariance.

In [ ]:
noise = read_mat("task41_xhat_noie_fligh.mat")
t_noise, measurement_noise = noise[0], noise[1:]
print(f"Task 1: {measurement_noise.shape[1]} samples, "
      f"{t_noise[-1]:.3f} s, finite={np.isfinite(measurement_noise).all()}")

fig, ax = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for k, a in enumerate(ax.flat):
    a.plot(t_noise, measurement_noise[k], lw=0.9)
    a.set_title(STATE_NAMES[k])
    a.set_ylabel(STATE_LABELS[k])
    a.set_xlabel("Time [s]")
fig.suptitle("Task 1: recorded signals during flight", fontsize=14)
fig.tight_layout()
save_figure(fig, "task1_flight_signals")

In [ ]:
Rd = np.cov(measurement_noise)  # 6 x N: rows are channels (as in MATLAB cov(X'))
print("Covariance estimated from the full flying recording:")
print(np.array2string(Rd, precision=6, suppress_small=False))
print("Diagonal variances:", np.array2string(np.diag(Rd), precision=6))

fig, ax = plt.subplots(figsize=(8.5, 7))
heatmap = ax.imshow(Rd, cmap="coolwarm")
fig.colorbar(heatmap, ax=ax, shrink=0.8, label="Covariance (mixed units)")
ax.set_xticks(range(6), STATE_NAMES, rotation=45, ha="right")
ax.set_yticks(range(6), STATE_NAMES)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{Rd[i,j]:.2g}", ha="center", va="center", fontsize=8)
ax.set_title(r"Task 1: empirical covariance $R_d$ (flight only)")
fig.tight_layout()
save_figure(fig, "task1_Rd_covariance")

### Stationarity check

Compare covariance diagonals from the first and second halves of the flight recording. This is **not** a substitute for the missing ground-versus-flight comparison. A large difference indicates that the full-record covariance is sensitive to the chosen interval.

In [ ]:
mid = measurement_noise.shape[1] // 2
var_first = np.var(measurement_noise[:, :mid], axis=1, ddof=1)
var_second = np.var(measurement_noise[:, mid:], axis=1, ddof=1)
print("First-half variances: ", np.round(var_first, 6))
print("Second-half variances:", np.round(var_second, 6))

idx = np.arange(6)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(idx - .18, var_first, .36, label="First half")
ax.bar(idx + .18, var_second, .36, label="Second half")
ax.set_yscale("log")
ax.set_xticks(idx, STATE_NAMES, rotation=25, ha="right")
ax.set_ylabel("Sample variance (log scale; channel units squared)")
ax.set_title("Task 1: variance changes within the flight recording")
ax.legend()
fig.tight_layout()
save_figure(fig, "task1_stationarity_check")

## 2. Task 2 — Kalman-filter implementation

For each available experiment, compare the Kalman estimate ($\hat x$), the transformed sensor output ($y$), and encoder values. Plots are split into **angles** and **angular rates** for readability. All recorded samples are shown without resampling, filtering or outlier removal.

The elevation encoder curve is corrected for the identified extra logging subtraction of `0.60 rad`; this does not correct possible IMU bias.

In [ ]:
def plot_state_comparison(run, suffix, q, indices, kind):
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for a, i in zip(ax, indices):
        t = run["t"]
        a.plot(t, run["y"][i], label="Sensor output y", alpha=0.65, lw=0.8)
        a.plot(t, run["encoder"][i], label="Encoder", lw=1.0, ls="--")
        a.plot(t, run["xhat"][i], label="Kalman estimate", lw=1.3)
        a.set_ylabel(STATE_LABELS[i])
        a.set_title(STATE_NAMES[i], loc="left", fontsize=10)
    ax[-1].set_xlabel("Time [s]")
    fig.suptitle(f"Task 2/3: {kind} — filename parameter q={q:g}", fontsize=13)
    handles, labels = ax[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(.5, .955))
    fig.tight_layout(rect=[0, 0, 1, .91])
    save_figure(fig, f"task2_{kind.lower().replace(' ', '_')}_q_{suffix}")

for suffix, q in CASES:
    plot_state_comparison(runs[suffix], suffix, q, [0, 2, 4], "Angles")
    plot_state_comparison(runs[suffix], suffix, q, [1, 3, 5], "Angular rates")

### Reference tracking

The recorded reference vector is $r=[p_c,\dot e_c]^T$. These graphs compare the reference with estimates and encoders for the controlled variables. Reference signals and test durations differ between experiments, so they are **not identical-condition step responses**.

In [ ]:
for suffix, q in CASES:
    run = runs[suffix]
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for a, state_i, ref_i in [(ax[0], 0, 0), (ax[1], 3, 1)]:
        a.plot(run["t"], run["reference"][ref_i], label="Reference", lw=1.5)
        a.plot(run["t"], run["encoder"][state_i], label="Encoder", lw=1.0, ls="--")
        a.plot(run["t"], run["xhat"][state_i], label="Kalman estimate", lw=1.2)
        a.set_ylabel(STATE_LABELS[state_i])
        a.legend(loc="best")
    ax[-1].set_xlabel("Time [s]")
    fig.suptitle(f"Task 3: pitch and elevation-rate tracking — q={q:g}")
    fig.tight_layout()
    save_figure(fig, f"task3_reference_tracking_q_{suffix}")

## 3. Task 3 — Effect of tuning $Q_d$

The experiment filenames indicate parameters `20`, `1`, `0.1`, `0.0001` and `0.00001`. The **actual matrices used for each run were not saved**, so these values are labels inferred from the filenames, not a verification of $Q_d = qI$.

Two diagnostics are useful:

1. RMS difference between estimator and sensor output: illustrates how closely the estimate follows raw measurements.
2. RMS difference relative to the encoders: a *cross-sensor agreement metric*, **not ground-truth estimation error**. The runs have different joystick inputs, flight conditions, durations and potential sensor biases. We therefore cannot rank estimator performance solely by these RMS values.

Travel angle is omitted from the tuning comparisons because its `y` measurement comes directly from the encoder.

In [ ]:
q_values = np.array([q for suffix, q in CASES])
keep = [0, 1, 2, 3, 5]  # Not travel angle: its y-channel is already the encoder.

fig, ax = plt.subplots(3, 2, figsize=(11, 9), sharex=True)
for a, i in zip(ax.flat, keep):
    values = [np.sqrt(np.mean((runs[s]["xhat"][i] - runs[s]["y"][i]) ** 2))
              for s, _ in CASES]
    a.plot(q_values, values, "o-")
    a.set_xscale("log")
    a.set_ylabel("RMS(estimate - y)")
    a.set_title(STATE_NAMES[i])
    a.grid(True, which="both", alpha=.2)
ax.flat[-1].axis("off")
for a in ax[-1, :]:
    a.set_xlabel("q (inferred from filenames)")
fig.suptitle("Task 3: Kalman estimate vs. sensor measurement")
fig.tight_layout()
save_figure(fig, "task3_estimate_measurement_difference")

fig, ax = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for a, i in zip(ax.flat, [0, 1, 2, 3]):
    rms_y = [np.sqrt(np.mean((runs[s]["y"][i] - runs[s]["encoder"][i]) ** 2))
             for s, _ in CASES]
    rms_x = [np.sqrt(np.mean((runs[s]["xhat"][i] - runs[s]["encoder"][i]) ** 2))
             for s, _ in CASES]
    a.plot(q_values, rms_y, "o--", label="Sensor vs encoder")
    a.plot(q_values, rms_x, "o-", label="Kalman vs encoder")
    a.set_xscale("log")
    a.set_ylabel("RMS difference [rad or rad/s]")
    a.set_title(STATE_NAMES[i])
    a.legend()
    a.grid(True, which="both", alpha=.2)
for a in ax[-1, :]:
    a.set_xlabel("q (inferred from filenames)")
fig.suptitle("Task 3: cross-sensor agreement (not true estimation error)")
fig.tight_layout()
save_figure(fig, "task3_encoder_agreement")

### Notable short rate spikes

The `qis20` and `qisp0001` datasets contain isolated high angular-rate values in the transformed measurements. They are not removed. The figure below shows a short interval around the most prominent spike in `qis20`; compare the raw output, Kalman estimate and encoder rates.

In [ ]:
run = runs["20"]
selection = (run["t"] >= 45.8) & (run["t"] <= 46.6)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for a, i in zip(ax, [1, 5]):
    a.plot(run["t"][selection], run["y"][i, selection], label="Sensor output y", lw=1.0)
    a.plot(run["t"][selection], run["xhat"][i, selection], label="Kalman estimate", lw=1.2)
    a.plot(run["t"][selection], run["encoder"][i, selection], label="Encoder", ls="--", lw=1.1)
    a.set_ylabel(STATE_LABELS[i])
    a.set_title(STATE_NAMES[i], loc="left")
    a.legend()
ax[-1].set_xlabel("Time [s]")
fig.suptitle("Task 3: isolated rate spikes, recording qis20")
fig.tight_layout()
save_figure(fig, "task3_rate_spikes_q20")

## 4. Missing information and report considerations

- **Task 1:** A stationary-ground dataset is missing. The only flight recording also exhibits changes in variance between intervals. A reliable, direct ground-versus-flight covariance comparison is therefore not possible.
- **Task 2:** The saved state trajectories show what the implemented Kalman filter produces. They do not, on their own, verify the internal equations for prediction/correction. The supplied Simulink contains those subsystems.
- **Task 3:** `new_data` and the filter covariance $\hat P$ were not exported as time-series. The requested experiment with disconnecting measurements cannot be documented quantitatively from these MAT files. Do not infer covariance growth from $\hat x$ alone.
- **Sensor limitations:** The `y` travel-angle channel is sourced from the encoder. The elevation encoder in the files has been logged with a duplicated subtraction of `0.60 rad`, and a smaller residual elevation discrepancy remains after correcting it.
- **Controller detail:** In the supplied `init_heli_3_10.m`, `K(:,4:5)` is overwritten with zeros after `lqr(...)`, disabling integral-state feedback. Whether this was intentional cannot be established from the recordings.

Refer to the accompanying `DATAKONTROLL.md` for the full diagnostic summary. All plots in this notebook use the existing measurements; no missing experiments are generated.